# Week 10b.1: Evaluation as an Experiment

In the last notebook, we ran 16 tasks once and scored each run. Suppose we now change the prompt, run the tasks again, and score one more task as passed. **Is the new prompt better?**

We cannot tell from one run of each. The same agent on the same task can pass on one run and fail on the next, so part of any difference between two runs is noise. To say that one version of an agent is better than another, an evaluation has to be run as an experiment:

- **Two or more conditions** that differ in one thing, one of which is a competent baseline
- **Several runs** of every task under every condition, saved to files
- **An estimate with its uncertainty**, such as a success rate with a confidence interval, and a comparison that accounts for the same tasks appearing in both conditions
- **The cost** of each condition, because a version that wins by spending twice as much has not shown the same thing as one that wins at equal cost
- **A study of the failures**, coded into categories by more than one person


In [ ]:
from dotenv import load_dotenv
import os

load_dotenv()
keys = ["OPENAI_API_KEY", "ANTHROPIC_API_KEY", "GOOGLE_API_KEY"]
print("API keys found:", [k for k in keys if os.getenv(k)] or "none (fine with Ollama or the saved results)")

## 0. Model Setup

Execute one of the following to define the model.

In [ ]:
from langchain_ollama import ChatOllama

model = ChatOllama(model="qwen3.5:9b", reasoning=False)

In [ ]:
from langchain.chat_models import init_chat_model

model = init_chat_model(model="gpt-4.1-mini")

In [ ]:
from langchain.chat_models import init_chat_model

model = init_chat_model(model="claude-haiku-5-5")

In [ ]:
from langchain_google_genai import ChatGoogleGenerativeAI

model = ChatGoogleGenerativeAI(model="gemini-3.5-flash-lite")

## 1. Two conditions

We'll compare two prompts for the support agent from W10a.1.

- **policy**: `husky.POLICY_PROMPT`, the prompt from the last notebook. This is the baseline, and it is a competent one: the policy is complete and clearly written.
- **check**: the same prompt plus an instruction to write out, for each order, its delivery age and price as the tools report them, and the rule that applies, before acting.

Our **hypothesis** is that writing the facts out first helps on the tasks where the customer's claim contradicts the records, such as t16, at the cost of more tokens per run. Everything else, including the model, the tools, the tasks, and the checkers, is the same in both conditions.

In [ ]:
import json
import time
from pathlib import Path

import husky_env as husky
from eval_harness import build_support, run_task, check_outcome, check_trajectory, cohen_kappa



## 2. The harness

`run_experiment` runs every task `N_RUNS` times under every condition, and appends each run's record to a file, one JSON object per line. Three things about it are worth copying into your own harness.

- **Every run is saved as soon as it finishes.** If the experiment stops halfway, through a 429 error, a crash, or a closed laptop, running the cell again skips the runs already in the file and continues.
- **The files hold raw records, not scores.** The checkers run when we analyze, so if we fix a checker, we rescore the saved runs instead of running the agent again.
- **The runs are interleaved.** The loop does run 0 of every task, then run 1 of every task, and so on. If the experiment stops early, every task has the same number of runs.

The tools share one environment, so tasks run one at a time.

In [ ]:
def load(path):
    return [json.loads(line) for line in path.read_text().splitlines()] if path.exists() else []

In [ ]:
def run_experiment(model, conditions, tasks, n_runs, folder, pause=0):
    folder.mkdir(parents=True, exist_ok=True)
    
    for name, prompt in conditions.items():
        # JSONL puts one JSON record on each line
        path = folder / f"{name}.jsonl"
        done = {(r["task"], r["run"]) for r in load(path)}
        agent = build_support(model, prompt)
        
        for run in range(n_runs):
            for task in tasks:
                if (task["id"], run) in done:
                    continue
                record = run_task(agent, task, config={"run_name": f"{name} {task['id']} run {run}",
                                                       "tags": ["w10b", name], "metadata": {"task": task["id"]}})
                record.update(condition=name, run=run)
                with open(path, "a") as f:
                    f.write(json.dumps(record) + "\n")
                time.sleep(pause)
            print(f"{name}: run {run} done")

The experiment is 16 tasks, 5 runs, and 2 conditions: 160 agent runs, each a few model calls. With gpt-4.1-mini it took us about six minutes. On the Gemini free tier, at 15 calls a minute, it takes over half an hour, so set `pause=4` to stay under the limit.

The results of our run with gpt-4.1-mini are saved in `results/gpt-4.1-mini/`, so you can do the analysis without running the experiment. To run it with your own model, set `RUN = True`. Your results go to a folder named after your model.

In [ ]:
CONDITIONS = {"policy": husky.POLICY_PROMPT, "check": husky.CHECK_PROMPT}
tasks = husky.tasks()
task_by_id = {t["id"]: t for t in tasks}

In [ ]:
RUN = False
N_RUNS = 5

model_name = (getattr(model, "model_name", None) or getattr(model, "model", "model")).split("/")[-1]
folder = Path("results") / (model_name if RUN else "gpt-4.1-mini")

if RUN:
    run_experiment(model, CONDITIONS, tasks, N_RUNS, folder, pause=0)

results = {name: load(folder / f"{name}.jsonl") for name in CONDITIONS}
for name, records in results.items():
    print(f"{name:7} {len(records)} runs from {folder}")

## 3. Results by task

For every run, we apply the outcome check from W10a.1. The table shows, for each task, how many of its runs passed under each condition.

In [ ]:
def passes(records):
    """For each task, the outcome of each run in run order."""
    by_task = {}
    for r in sorted(records, key=lambda r: r["run"]):
        by_task.setdefault(r["task"], []).append(check_outcome(task_by_id[r["task"]], r)[0])
    return by_task


outcomes = {name: passes(records) for name, records in results.items()}

print(f"{'task':6}{'category':20}" + "".join(f"{name:>10}" for name in CONDITIONS))
for t in tasks:
    row = f"{t['id']:6}{t['category']:20}"
    for name in CONDITIONS:
        runs = outcomes[name][t["id"]]
        row += f"{sum(runs)}/{len(runs)}".rjust(10)
    print(row)

Look for three kinds of task.

- **Tasks that always pass** under both conditions. They confirm the agent handles the basics, but they cannot show a difference between the conditions.
- **Tasks that pass on some runs and fail on others.** These are where run-to-run variation lives. With a single run, such a task counts as a pass or a fail depending on luck.
- **Tasks that always fail.** Check these by hand first. A task that fails every run under every condition may have a wrong expected state.

## 4. One number with its uncertainty

The success rate is the mean over tasks of each task's pass rate, so every task counts equally however many runs it has. Two other summaries are useful for agents:

- **pass^k** is the fraction of tasks that passed on all k runs. τ-bench introduced it to measure reliability. 
    - An agent that passes every task 80% of the time has a success rate of 0.8, but its pass^5 is much lower, because few tasks pass five times in a row. 
    - A customer-facing agent needs the second number to be high.
- **The 95% confidence interval** says how far the success rate could move if we had drawn a different set of tasks of the same kind. 
    - We compute it with the **bootstrap**: resample the 16 tasks with replacement many times, recompute the success rate each time, and take the middle 95% of the results. The tasks, not the runs, are the unit we resample, because the tasks are what we want to generalize over.

In [ ]:
import random


def bootstrap_ci(values, n=10_000, seed=0):
    rng = random.Random(seed)
    means = sorted(sum(rng.choices(values, k=len(values))) / len(values) for _ in range(n))
    return means[int(0.025 * n)], means[int(0.975 * n)]


def summary(by_task):
    rates = [sum(runs) / len(runs) for runs in by_task.values()]
    low, high = bootstrap_ci(rates)
    pass_k = sum(all(runs) for runs in by_task.values()) / len(by_task)
    return sum(rates) / len(rates), low, high, pass_k


print(f"{'condition':10}{'success':>9}{'95% CI':>16}{'pass^5':>9}")
for name in CONDITIONS:
    rate, low, high, pass_k = summary(outcomes[name])
    print(f"{name:10}{rate:>9.2f}    [{low:.2f}, {high:.2f}]{pass_k:>9.2f}")

## 5. Comparing the conditions

The two intervals above overlap, but that alone does not settle the question, and it is not the right comparison. Both conditions ran the same 16 tasks, so we can compare them task by task: 
- for each task, the difference between its pass rate under `check` and under `policy`. 

Easy tasks contribute a difference of zero, and the comparison concentrates on the tasks where the conditions behave differently. This is a **paired** comparison, and it is more sensitive than comparing two separate intervals.

We bootstrap the mean of the per-task differences. If the interval includes zero, the experiment has not shown a difference.

In [ ]:
differences = []
for t in tasks:
    a, b = outcomes["policy"][t["id"]], outcomes["check"][t["id"]]
    differences.append(sum(b) / len(b) - sum(a) / len(a))

mean_difference = sum(differences) / len(differences)
low, high = bootstrap_ci(differences)
print(f"check minus policy: {mean_difference:+.3f}   95% CI [{low:+.3f}, {high:+.3f}]")
print("tasks that differ: ", {t["id"]: round(d, 1) for t, d in zip(tasks, differences) if d != 0})

### Cost

Now what each condition spent. We report tokens and time per run, and tokens per passed run, which puts cost and success in one number.

In [ ]:
print(f"{'condition':10}{'tokens/run':>12}{'seconds/run':>13}{'tokens per pass':>17}")
for name, records in results.items():
    tokens = sum(r["tokens"] for r in records)
    seconds = sum(r["seconds"] for r in records)
    passed = sum(check_outcome(task_by_id[r["task"]], r)[0] for r in records)
    print(f"{name:10}{tokens / len(records):>12.0f}{seconds / len(records):>13.1f}{tokens / max(passed, 1):>17.0f}")

In the saved gpt-4.1-mini results, the `check` prompt passed t16 on all five runs, where the baseline failed all five, and it passed t14 on four runs against two. It also failed t12 once, which the baseline never did. The mean difference is positive, but its interval runs from slightly below zero upward, because only three of the 16 tasks differ at all. The `check` prompt costs about a fifth more tokens per run.

Read a result like this carefully before you write a conclusion.

- **The effect on t16 is large and consistent**, five passes against none, so the prompt very likely changes how the agent handles that task.
- **The experiment has not shown that the prompt improves the agent overall.** Thirteen tasks never differed, and the interval for the mean difference includes zero. Do not round it up to "slightly better".
- **The next step is a planned follow-up**, with more tasks of the kinds where the conditions differed, such as false claims and injected instructions, decided before you see the new results. That is the fifth item in Things to try.

If the follow-up still shows no difference, that is worth reporting too. It tells the next person that this change, on these tasks, with this model, did not buy enough to be seen, and saves them the tokens. 

## 6. How many tasks?

With 16 tasks, the intervals are wide. The width of a 95% interval for a success rate p over N independent tasks is roughly ±1.96·√(p(1−p)/N).

 The table shows the half-width for a few sizes.

In [ ]:
print(f"{'tasks':>6}" + "".join(f"{'p = ' + str(p):>12}" for p in (0.95, 0.85, 0.7)))
for n in (16, 50, 100, 200, 400):
    print(f"{n:>6}" + "".join(f"{'±' + format(1.96 * (p * (1 - p) / n) ** 0.5, '.3f'):>12}" for p in (0.95, 0.85, 0.7)))

To see a difference of five percentage points between two conditions, the intervals have to be narrower than that, which takes a few hundred tasks. Repeated runs help with the variation within a task, but they do not replace tasks: five runs of 16 tasks tell you a lot about those 16 tasks and little about the 17th.

## 7. Failure analysis

A success rate says how often the agent fails. Failure analysis says how, which is what tells you what to change. The method is the one we used for the judge's labels:

1. Collect every failed run, with its message, the evidence, and the reply.
2. Write a short list of failure categories, a taxonomy, from a first pass through the failures.
3. Have two people assign a category to every failure independently.
4. Report their agreement as kappa, and settle the disagreements by discussion. Where they disagree often, the categories need clearer definitions.

Here are the failures from both conditions, grouped by task and problem.

In [ ]:
failures = []
for name, records in results.items():
    for r in records:
        ok, problem = check_outcome(task_by_id[r["task"]], r)
        if not ok:
            failures.append({"condition": name, "task": r["task"], "run": r["run"], "problem": problem,
                             "message": task_by_id[r["task"]]["message"], "reply": r["reply"], "evidence": r["evidence"]})

groups = {}
for f in failures:
    groups.setdefault((f["task"], f["problem"]), []).append(f["condition"])
for (task_id, problem), conds in sorted(groups.items()):
    print(f"{task_id}  {problem:60} policy {conds.count('policy')}  check {conds.count('check')}")

For this agent, a first taxonomy might be:

| Code | Meaning |
|---|---|
| `trusted-customer` | used a fact the customer stated instead of the records |
| `followed-injection` | followed an instruction in the message that contradicts the policy |
| `misapplied-rule` | had the right facts and applied the wrong rule |
| `missed-request` | handled one request in the message and ignored another |
| `other` | anything else, with a note |

The next cell writes the failures to `failures.csv` with two empty columns, `code_a` and `code_b`. Open it in a spreadsheet, and have two people fill in one column each without looking at the other.

In [ ]:
import csv

with open("failures.csv", "w", newline="") as f:
    writer = csv.DictWriter(f, fieldnames=["condition", "task", "run", "problem", "message", "reply", "evidence",
                                           "code_a", "code_b"])
    writer.writeheader()
    for row in failures:
        writer.writerow({**row, "code_a": "", "code_b": ""})

print(f"wrote {len(failures)} failures to failures.csv")

When both columns are filled in, this cell reports the agreement and lists the disagreements to discuss.

In [ ]:
with open("failures.csv") as f:
    coded = [row for row in csv.DictReader(f) if row["code_a"] and row["code_b"]]

if coded:
    a, b = [row["code_a"] for row in coded], [row["code_b"] for row in coded]
    print(f"{len(coded)} failures coded, kappa = {cohen_kappa(a, b):.2f}")
    for row in coded:
        if row["code_a"] != row["code_b"]:
            print(f"  {row['task']} run {row['run']}: {row['code_a']} vs {row['code_b']}")
else:
    print("No coded rows yet. Fill in code_a and code_b in failures.csv, then run this cell again.")

## 8. The same evaluation in LangSmith

LangSmith runs the same kind of evaluation and keeps the results, so you can compare experiments side by side in its interface and open the trace of any run. The pieces map onto ours directly:

| Ours | LangSmith |
|---|---|
| `eval_tasks.json` | a dataset of examples, each with inputs and reference outputs |
| `run_task` | the target, a function from an example's inputs to outputs |
| `check_outcome`, `check_trajectory` | evaluators, functions that score the outputs against the reference outputs |
| `N_RUNS` | `num_repetitions` |
| the `.jsonl` files | an experiment |

This section sends data to LangSmith: it creates a dataset called `husky-refund-tasks` in your workspace, and `evaluate` sends a trace of every run, so it runs only if you set `USE_LANGSMITH = True`. It runs the baseline on all 16 tasks twice, which is 32 agent runs.

`max_concurrency=0` makes the examples run one at a time. LangSmith runs examples in parallel by default, and our tools share one environment, so parallel runs would mix their refunds together.

In [ ]:
from langsmith import Client, evaluate

USE_LANGSMITH = False

baseline = build_support(model, husky.POLICY_PROMPT)


def target(inputs):
    return run_task(baseline, task_by_id[inputs["task"]])


def outcome(outputs, reference_outputs):
    return check_outcome(reference_outputs, outputs)[0]


def trajectory(outputs):
    return check_trajectory(outputs)[0]


if USE_LANGSMITH:
    client = Client()
    if not client.has_dataset(dataset_name="husky-refund-tasks"):
        client.create_dataset("husky-refund-tasks")
        client.create_examples(dataset_name="husky-refund-tasks",
                               examples=[{"inputs": {"task": t["id"], "message": t["message"]}, "outputs": t}
                                         for t in tasks])
    experiment = evaluate(target, data="husky-refund-tasks", evaluators=[outcome, trajectory],
                          experiment_prefix="policy", num_repetitions=2, max_concurrency=0)
    for row in experiment:
        scores = {e.key: e.score for e in row["evaluation_results"]["results"]}
        print(row["example"].inputs["task"], scores)
else:
    print("Skipped. Set USE_LANGSMITH = True to send the dataset and the runs to your LangSmith workspace.")

## 9. A checklist for your evaluation

| | Question | Where it went today |
|---|---|---|
| Tasks | Where do they come from, how many, which categories, which are held out? | 16 tasks over 13 categories in `eval_tasks.json` |
| Ground truth | Is it the final state of the environment, a deterministic check, or a judge? | the environment's refunds and escalations |
| Judge | If you use one, what is its kappa with human labels, on how many items? | 18 labeled replies in W10a.1 |
| Baseline | Is it what a capable practitioner would do without your method? | the complete policy prompt |
| Runs | How many per task and condition, with which sampling settings? | 5 runs, default settings |
| Uncertainty | Which interval, over which unit? | bootstrap over tasks, paired for the comparison |
| Cost | Tokens, dollars, or time for every condition? | tokens and seconds per run, tokens per pass |
| Failures | Which taxonomy, coded by whom, with what agreement? | `failures.csv`, two coders, kappa |
| Records | Can you rescore the runs without rerunning them? | one JSON line per run |


## 10. Things to try

1. Run the experiment with your own model (`RUN = True`). Is the baseline's success rate inside the interval we computed for gpt-4.1-mini? Which tasks change?

2. Compare two models instead of two prompts. Use the same `CONDITIONS` dictionary, but build each condition's agent with a different model. Which comparison is the fairer one if the two models have different prices per token?

3. Reduce `N_RUNS` to 1 and recompute the table in section 3 and the comparison in section 5 using only run 0. Would you have drawn the same conclusion? Repeat for run 1.

4. Code the failures in `failures.csv` with a classmate and compute kappa. Which category did you disagree on most, and how would you reword its definition?

5. Write five new tasks in the category where the conditions differed most, add them to `eval_tasks.json`, and rerun only those tasks. Does the difference between the conditions grow, shrink, or disappear?

In [ ]:
# your experiments here